<a href="https://colab.research.google.com/github/jaehyeon0420/agent_tutorial/blob/master/rag_basic_example/rag_function_calling_bread.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 본 파일은 가상 데이터 기반으로 Function Calling 또는 Text-to-SQL을 진행한 자료입니다.

In [1]:
!pip install -q langchain langgraph openai langchain_openai python-dotenv langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
from google.colab import userdata
import os
import sqlite3
import tempfile
from langchain_community.utilities import SQLDatabase

In [3]:
os.environ["OPENAI_API_KEY"] = userdata.get('OPEN_AI_KEY')

#### 테스트를 위한 사전 합성 데이터를 생성합니다.

In [5]:
bakery_schema = """
-- 매장 테이블
CREATE TABLE stores (
    store_id VARCHAR(10) PRIMARY KEY,      -- 매장ID (예: S001)
    store_name VARCHAR(100) NOT NULL,      -- 매장명
    address VARCHAR(200) NOT NULL,         -- 주소
    city VARCHAR(50) NOT NULL,             -- 도시
    phone VARCHAR(20) NOT NULL,            -- 전화번호
    opened_date DATE NOT NULL              -- 개점일자
);

-- 카테고리 테이블
CREATE TABLE categories (
    category_id VARCHAR(10) PRIMARY KEY,   -- 카테고리ID (예: CAT001)
    category_name VARCHAR(50) NOT NULL,    -- 카테고리명
    description VARCHAR(200)               -- 설명
);

-- 빵 상품 테이블
CREATE TABLE products (
    product_id VARCHAR(10) PRIMARY KEY,    -- 상품ID (예: P001)
    product_name VARCHAR(100) NOT NULL,    -- 상품명
    category_id VARCHAR(10) NOT NULL,      -- 카테고리ID
    price INTEGER NOT NULL,                -- 가격
    description VARCHAR(300),              -- 설명
    image_url VARCHAR(200),                -- 이미지URL
    FOREIGN KEY(category_id) REFERENCES categories(category_id)
);

-- 매장 운영시간 테이블
CREATE TABLE store_hours (
    store_hours_id VARCHAR(10) PRIMARY KEY, -- 운영시간ID (예: SH001)
    store_id VARCHAR(10) NOT NULL,          -- 매장ID
    day_of_week VARCHAR(20) NOT NULL,       -- 요일 (월,화,수,목,금,토,일)
    opening_time VARCHAR(8) NOT NULL,       -- 오픈시간 (HH:MM:SS)
    closing_time VARCHAR(8) NOT NULL,       -- 마감시간 (HH:MM:SS)
    FOREIGN KEY(store_id) REFERENCES stores(store_id)
);

-- 재고 테이블
CREATE TABLE inventory (
    inventory_id VARCHAR(10) PRIMARY KEY,  -- 재고ID (예: INV001)
    store_id VARCHAR(10) NOT NULL,         -- 매장ID
    product_id VARCHAR(10) NOT NULL,       -- 상품ID
    quantity_in_stock INTEGER NOT NULL,    -- 재고수량
    last_updated DATETIME NOT NULL,        -- 마지막 업데이트 시간
    FOREIGN KEY(store_id) REFERENCES stores(store_id),
    FOREIGN KEY(product_id) REFERENCES products(product_id)
);

-- 알레르겐 테이블
CREATE TABLE allergens (
    allergen_id VARCHAR(10) PRIMARY KEY,   -- 알레르겐ID (예: ALG001)
    allergen_name VARCHAR(50) NOT NULL     -- 알레르겐명 (계란, 우유, 견과류 등)
);

-- 상품-알레르겐 매핑 테이블
CREATE TABLE product_allergens (
    product_allergen_id VARCHAR(10) PRIMARY KEY, -- ID (예: PA001)
    product_id VARCHAR(10) NOT NULL,             -- 상품ID
    allergen_id VARCHAR(10) NOT NULL,            -- 알레르겐ID
    FOREIGN KEY(product_id) REFERENCES products(product_id),
    FOREIGN KEY(allergen_id) REFERENCES allergens(allergen_id)
);

-- 프로모션/할인 테이블
CREATE TABLE promotions (
    promotion_id VARCHAR(10) PRIMARY KEY,  -- 프로모션ID (예: PROMO001)
    product_id VARCHAR(10) NOT NULL,       -- 상품ID
    store_id VARCHAR(10),                  -- 매장ID (NULL이면 전체 매장 적용)
    discount_percent REAL NOT NULL,        -- 할인율 (%)
    valid_from DATE NOT NULL,              -- 할인 시작일
    valid_until DATE NOT NULL,             -- 할인 종료일
    FOREIGN KEY(product_id) REFERENCES products(product_id),
    FOREIGN KEY(store_id) REFERENCES stores(store_id)
);

-- 리뷰 테이블 (보너스)
CREATE TABLE reviews (
    review_id VARCHAR(10) PRIMARY KEY,     -- 리뷰ID (예: R001)
    product_id VARCHAR(10) NOT NULL,       -- 상품ID
    rating INTEGER NOT NULL,               -- 평점 (1~5)
    comment VARCHAR(500),                  -- 리뷰 내용
    review_date DATE NOT NULL,             -- 리뷰 작성일
    FOREIGN KEY(product_id) REFERENCES products(product_id)
);
"""

In [6]:
# 매장 샘플 데이터
sample_stores = """
INSERT INTO stores VALUES('S001', '서울 강남점', '서울시 강남구 강남대로 123', '서울', '02-1234-5678', '2018-03-15');
INSERT INTO stores VALUES('S002', '서울 명동점', '서울시 중구 명동 45-6', '서울', '02-2345-6789', '2019-06-20');
INSERT INTO stores VALUES('S003', '부산 해운대점', '부산시 해운대구 해운대로 789', '부산', '051-3456-7890', '2017-11-10');
INSERT INTO stores VALUES('S004', '대구 중앙점', '대구시 중구 달성로 234', '대구', '053-4567-8901', '2020-01-05');
INSERT INTO stores VALUES('S005', '인천 연수점', '인천시 연수구 청량로 567', '인천', '032-5678-9012', '2021-08-18');
INSERT INTO stores VALUES('S006', '대전 둔산점', '대전시 서구 둔산로 891', '대전', '042-6789-0123', '2019-05-22');
INSERT INTO stores VALUES('S007', '광주 수완점', '광주시 광산구 수완로 112', '광주', '062-7890-1234', '2022-02-14');
"""

# 카테고리 샘플 데이터
sample_categories = """
INSERT INTO categories VALUES('CAT001', '식빵', '슬라이스 식빵 및 통곡물빵');
INSERT INTO categories VALUES('CAT002', '롤빵', '크림 롤빵, 초콜릿 롤빵 등');
INSERT INTO categories VALUES('CAT003', '크로와상', '버터 크로와상 및 초콜릿 크로와상');
INSERT INTO categories VALUES('CAT004', '케이크', '생크림케이크, 초콜릿케이크 등');
INSERT INTO categories VALUES('CAT005', '베이글', '플레인 베이글, 블루베리 베이글 등');
INSERT INTO categories VALUES('CAT006', '쿠키/마카롱', '수제쿠키 및 마카롱');
"""

# 빵 상품 샘플 데이터
sample_products = """
INSERT INTO products VALUES('P001', '프리미엄 버터 식빵', 'CAT001', 5500, '신선한 버터와 우유로 만든 식빵', 'img_butter_bread.jpg');
INSERT INTO products VALUES('P002', '통곡물 건강식빵', 'CAT001', 6000, '귀리와 현미가 들어간 건강식빵', 'img_whole_grain.jpg');
INSERT INTO products VALUES('P003', '초코칩 식빵', 'CAT001', 5800, '초코칩이 가득한 달콤한 식빵', 'img_choco_bread.jpg');
INSERT INTO products VALUES('P004', '슈가 롤빵', 'CAT002', 3500, '바삭한 설탕 코팅 롤빵', 'img_sugar_roll.jpg');
INSERT INTO products VALUES('P005', '초콜릿 크림 롤빵', 'CAT002', 4200, '초콜릿 크림이 가득한 롤빵', 'img_choco_roll.jpg');
INSERT INTO products VALUES('P006', '딸기 크림 롤빵', 'CAT002', 4500, '신선한 딸기와 생크림 롤빵', 'img_strawberry_roll.jpg');
INSERT INTO products VALUES('P007', '버터 크로와상', 'CAT003', 4000, '황금빛 버터 크로와상', 'img_butter_croissant.jpg');
INSERT INTO products VALUES('P008', '초콜릿 크로와상', 'CAT003', 4300, '진한 초콜릿이 들어간 크로와상', 'img_choco_croissant.jpg');
INSERT INTO products VALUES('P009', '아몬드 크로와상', 'CAT003', 4800, '통 아몬드가 얹혀진 크로와상', 'img_almond_croissant.jpg');
INSERT INTO products VALUES('P010', '생크림 케이크', 'CAT004', 35000, '신선한 생크림과 과일로 장식한 케이크', 'img_cream_cake.jpg');
INSERT INTO products VALUES('P011', '초콜릿 가나슈 케이크', 'CAT004', 38000, '진한 초콜릿 가나슈 케이크', 'img_ganache_cake.jpg');
INSERT INTO products VALUES('P012', '딸기 타르트', 'CAT004', 28000, '신선한 딸기와 카스터드 타르트', 'img_strawberry_tart.jpg');
INSERT INTO products VALUES('P013', '플레인 베이글', 'CAT005', 3500, '기본이 되는 플레인 베이글', 'img_plain_bagel.jpg');
INSERT INTO products VALUES('P014', '블루베리 베이글', 'CAT005', 4000, '블루베리가 가득한 베이글', 'img_blueberry_bagel.jpg');
INSERT INTO products VALUES('P015', '에브리씬 베이글', 'CAT005', 3800, '온갖 곡물이 들어간 베이글', 'img_everything_bagel.jpg');
INSERT INTO products VALUES('P016', '초콜릿 칩 쿠키', 'CAT006', 2500, '초콜릿 칩이 가득한 쿠키', 'img_choco_cookie.jpg');
INSERT INTO products VALUES('P017', '더블 초콜릿 쿠키', 'CAT006', 3000, '진한 초콜릿 쿠키', 'img_double_choco_cookie.jpg');
INSERT INTO products VALUES('P018', '마카롱 세트 (6개)', 'CAT006', 15000, '다양한 맛의 프랑스식 마카롱', 'img_macarons.jpg');
INSERT INTO products VALUES('P019', '소금 버터 카라멜 베이글', 'CAT005', 4300, '짭짤한 소금과 버터 카라멜', 'img_salted_caramel_bagel.jpg');
INSERT INTO products VALUES('P020', '호두 바나나 빵', 'CAT001', 6500, '호두와 바나나가 들어간 빵', 'img_walnut_banana.jpg');
INSERT INTO products VALUES('P021', '레몬 파운드 케이크', 'CAT004', 22000, '상큼한 레몬 향이 가득한 파운드 케이크', 'img_lemon_cake.jpg');
INSERT INTO products VALUES('P022', '딸기 마카롱', 'CAT006', 1800, '딸기 향의 개별 마카롱', 'img_strawberry_macaron.jpg');
INSERT INTO products VALUES('P023', '말차 라떼 롤빵', 'CAT002', 4800, '말차 크림과 딸기를 곁들인 롤빵', 'img_matcha_roll.jpg');
INSERT INTO products VALUES('P024', '앙금 식빵', 'CAT001', 5300, '팥 앙금이 들어간 식빵', 'img_red_bean_bread.jpg');
INSERT INTO products VALUES('P025', '까망베르 치즈 크로와상', 'CAT003', 5200, '프랑스 치즈가 들어간 크로와상', 'img_cheese_croissant.jpg');
INSERT INTO products VALUES('P026', '라즈베리 타르트', 'CAT004', 28500, '신선한 라즈베리 타르트', 'img_raspberry_tart.jpg');
INSERT INTO products VALUES('P027', '오트밀 클러스터 쿠키', 'CAT006', 3500, '오트밀과 견과류 쿠키', 'img_oatmeal_cookie.jpg');
INSERT INTO products VALUES('P028', '바나나 초콜릿 머핀', 'CAT002', 4000, '바나나와 초콜릿 머핀', 'img_banana_muffin.jpg');
INSERT INTO products VALUES('P029', '블랙포레스트 케이크', 'CAT004', 42000, '검은숲 케이크 - 체리와 생크림', 'img_blackforest_cake.jpg');
INSERT INTO products VALUES('P030', '피스타치오 마카롱', 'CAT006', 2000, '피스타치오 향의 마카롱', 'img_pistachio_macaron.jpg');
"""

# 운영시간 샘플 데이터
sample_store_hours = """
INSERT INTO store_hours VALUES('SH001', 'S001', '월', '07:00:00', '22:00:00');
INSERT INTO store_hours VALUES('SH002', 'S001', '화', '07:00:00', '22:00:00');
INSERT INTO store_hours VALUES('SH003', 'S001', '수', '07:00:00', '22:00:00');
INSERT INTO store_hours VALUES('SH004', 'S001', '목', '07:00:00', '22:00:00');
INSERT INTO store_hours VALUES('SH005', 'S001', '금', '07:00:00', '22:00:00');
INSERT INTO store_hours VALUES('SH006', 'S001', '토', '08:00:00', '21:00:00');
INSERT INTO store_hours VALUES('SH007', 'S001', '일', '09:00:00', '20:00:00');
INSERT INTO store_hours VALUES('SH008', 'S002', '월', '08:00:00', '23:00:00');
INSERT INTO store_hours VALUES('SH009', 'S002', '화', '08:00:00', '23:00:00');
INSERT INTO store_hours VALUES('SH010', 'S002', '수', '08:00:00', '23:00:00');
INSERT INTO store_hours VALUES('SH011', 'S002', '목', '08:00:00', '23:00:00');
INSERT INTO store_hours VALUES('SH012', 'S002', '금', '08:00:00', '23:00:00');
INSERT INTO store_hours VALUES('SH013', 'S002', '토', '08:00:00', '23:00:00');
INSERT INTO store_hours VALUES('SH014', 'S002', '일', '09:00:00', '22:00:00');
INSERT INTO store_hours VALUES('SH015', 'S003', '월', '07:00:00', '21:00:00');
INSERT INTO store_hours VALUES('SH016', 'S003', '화', '07:00:00', '21:00:00');
INSERT INTO store_hours VALUES('SH017', 'S003', '수', '07:00:00', '21:00:00');
INSERT INTO store_hours VALUES('SH018', 'S003', '목', '07:00:00', '21:00:00');
INSERT INTO store_hours VALUES('SH019', 'S003', '금', '07:00:00', '21:00:00');
INSERT INTO store_hours VALUES('SH020', 'S003', '토', '08:00:00', '20:00:00');
INSERT INTO store_hours VALUES('SH021', 'S003', '일', '09:00:00', '19:00:00');
INSERT INTO store_hours VALUES('SH022', 'S004', '월', '06:30:00', '21:00:00');
INSERT INTO store_hours VALUES('SH023', 'S004', '화', '06:30:00', '21:00:00');
INSERT INTO store_hours VALUES('SH024', 'S004', '수', '06:30:00', '21:00:00');
INSERT INTO store_hours VALUES('SH025', 'S004', '목', '06:30:00', '21:00:00');
INSERT INTO store_hours VALUES('SH026', 'S004', '금', '06:30:00', '21:00:00');
INSERT INTO store_hours VALUES('SH027', 'S004', '토', '08:00:00', '20:00:00');
INSERT INTO store_hours VALUES('SH028', 'S004', '일', '09:00:00', '19:00:00');
INSERT INTO store_hours VALUES('SH029', 'S005', '월', '07:00:00', '22:00:00');
INSERT INTO store_hours VALUES('SH030', 'S005', '화', '07:00:00', '22:00:00');
INSERT INTO store_hours VALUES('SH031', 'S005', '수', '07:00:00', '22:00:00');
INSERT INTO store_hours VALUES('SH032', 'S005', '목', '07:00:00', '22:00:00');
INSERT INTO store_hours VALUES('SH033', 'S005', '금', '07:00:00', '22:00:00');
INSERT INTO store_hours VALUES('SH034', 'S005', '토', '08:00:00', '21:00:00');
INSERT INTO store_hours VALUES('SH035', 'S005', '일', '09:00:00', '20:00:00');
INSERT INTO store_hours VALUES('SH036', 'S006', '월', '07:00:00', '21:00:00');
INSERT INTO store_hours VALUES('SH037', 'S006', '화', '07:00:00', '21:00:00');
INSERT INTO store_hours VALUES('SH038', 'S006', '수', '07:00:00', '21:00:00');
INSERT INTO store_hours VALUES('SH039', 'S006', '목', '07:00:00', '21:00:00');
INSERT INTO store_hours VALUES('SH040', 'S006', '금', '07:00:00', '21:00:00');
INSERT INTO store_hours VALUES('SH041', 'S006', '토', '08:00:00', '20:00:00');
INSERT INTO store_hours VALUES('SH042', 'S006', '일', '09:00:00', '19:00:00');
INSERT INTO store_hours VALUES('SH043', 'S007', '월', '08:00:00', '21:00:00');
INSERT INTO store_hours VALUES('SH044', 'S007', '화', '08:00:00', '21:00:00');
INSERT INTO store_hours VALUES('SH045', 'S007', '수', '08:00:00', '21:00:00');
INSERT INTO store_hours VALUES('SH046', 'S007', '목', '08:00:00', '21:00:00');
INSERT INTO store_hours VALUES('SH047', 'S007', '금', '08:00:00', '21:00:00');
INSERT INTO store_hours VALUES('SH048', 'S007', '토', '09:00:00', '20:00:00');
INSERT INTO store_hours VALUES('SH049', 'S007', '일', '10:00:00', '19:00:00');
"""

# 알레르겐 샘플 데이터
sample_allergens = """
INSERT INTO allergens VALUES('ALG001', '계란');
INSERT INTO allergens VALUES('ALG002', '우유');
INSERT INTO allergens VALUES('ALG003', '견과류');
INSERT INTO allergens VALUES('ALG004', '밀가루');
INSERT INTO allergens VALUES('ALG005', '호두');
INSERT INTO allergens VALUES('ALG006', '땅콩');
"""

# 상품-알레르겐 매핑 샘플 데이터
sample_product_allergens = """
INSERT INTO product_allergens VALUES('PA001', 'P001', 'ALG001');
INSERT INTO product_allergens VALUES('PA002', 'P001', 'ALG002');
INSERT INTO product_allergens VALUES('PA003', 'P001', 'ALG004');
INSERT INTO product_allergens VALUES('PA004', 'P002', 'ALG001');
INSERT INTO product_allergens VALUES('PA005', 'P002', 'ALG002');
INSERT INTO product_allergens VALUES('PA006', 'P002', 'ALG004');
INSERT INTO product_allergens VALUES('PA007', 'P003', 'ALG001');
INSERT INTO product_allergens VALUES('PA008', 'P003', 'ALG002');
INSERT INTO product_allergens VALUES('PA009', 'P003', 'ALG004');
INSERT INTO product_allergens VALUES('PA010', 'P004', 'ALG001');
INSERT INTO product_allergens VALUES('PA011', 'P004', 'ALG002');
INSERT INTO product_allergens VALUES('PA012', 'P004', 'ALG004');
INSERT INTO product_allergens VALUES('PA013', 'P005', 'ALG001');
INSERT INTO product_allergens VALUES('PA014', 'P005', 'ALG002');
INSERT INTO product_allergens VALUES('PA015', 'P005', 'ALG004');
INSERT INTO product_allergens VALUES('PA016', 'P006', 'ALG001');
INSERT INTO product_allergens VALUES('PA017', 'P006', 'ALG002');
INSERT INTO product_allergens VALUES('PA018', 'P006', 'ALG004');
INSERT INTO product_allergens VALUES('PA019', 'P007', 'ALG001');
INSERT INTO product_allergens VALUES('PA020', 'P007', 'ALG002');
INSERT INTO product_allergens VALUES('PA021', 'P007', 'ALG004');
INSERT INTO product_allergens VALUES('PA022', 'P008', 'ALG001');
INSERT INTO product_allergens VALUES('PA023', 'P008', 'ALG002');
INSERT INTO product_allergens VALUES('PA024', 'P008', 'ALG004');
INSERT INTO product_allergens VALUES('PA025', 'P009', 'ALG001');
INSERT INTO product_allergens VALUES('PA026', 'P009', 'ALG002');
INSERT INTO product_allergens VALUES('PA027', 'P009', 'ALG003');
INSERT INTO product_allergens VALUES('PA028', 'P009', 'ALG005');
INSERT INTO product_allergens VALUES('PA029', 'P010', 'ALG001');
INSERT INTO product_allergens VALUES('PA030', 'P010', 'ALG002');
INSERT INTO product_allergens VALUES('PA031', 'P010', 'ALG004');
INSERT INTO product_allergens VALUES('PA032', 'P011', 'ALG001');
INSERT INTO product_allergens VALUES('PA033', 'P011', 'ALG002');
INSERT INTO product_allergens VALUES('PA034', 'P011', 'ALG004');
INSERT INTO product_allergens VALUES('PA035', 'P012', 'ALG001');
INSERT INTO product_allergens VALUES('PA036', 'P012', 'ALG002');
INSERT INTO product_allergens VALUES('PA037', 'P012', 'ALG004');
INSERT INTO product_allergens VALUES('PA038', 'P013', 'ALG001');
INSERT INTO product_allergens VALUES('PA039', 'P013', 'ALG002');
INSERT INTO product_allergens VALUES('PA040', 'P013', 'ALG004');
INSERT INTO product_allergens VALUES('PA041', 'P014', 'ALG001');
INSERT INTO product_allergens VALUES('PA042', 'P014', 'ALG002');
INSERT INTO product_allergens VALUES('PA043', 'P014', 'ALG004');
INSERT INTO product_allergens VALUES('PA044', 'P015', 'ALG001');
INSERT INTO product_allergens VALUES('PA045', 'P015', 'ALG002');
INSERT INTO product_allergens VALUES('PA046', 'P015', 'ALG003');
INSERT INTO product_allergens VALUES('PA047', 'P016', 'ALG001');
INSERT INTO product_allergens VALUES('PA048', 'P016', 'ALG002');
INSERT INTO product_allergens VALUES('PA049', 'P016', 'ALG004');
INSERT INTO product_allergens VALUES('PA050', 'P017', 'ALG001');
INSERT INTO product_allergens VALUES('PA051', 'P017', 'ALG002');
INSERT INTO product_allergens VALUES('PA052', 'P017', 'ALG004');
INSERT INTO product_allergens VALUES('PA053', 'P018', 'ALG001');
INSERT INTO product_allergens VALUES('PA054', 'P018', 'ALG002');
INSERT INTO product_allergens VALUES('PA055', 'P018', 'ALG004');
INSERT INTO product_allergens VALUES('PA056', 'P019', 'ALG001');
INSERT INTO product_allergens VALUES('PA057', 'P019', 'ALG002');
INSERT INTO product_allergens VALUES('PA058', 'P019', 'ALG004');
INSERT INTO product_allergens VALUES('PA059', 'P020', 'ALG001');
INSERT INTO product_allergens VALUES('PA060', 'P020', 'ALG002');
INSERT INTO product_allergens VALUES('PA061', 'P020', 'ALG003');
INSERT INTO product_allergens VALUES('PA062', 'P020', 'ALG004');
INSERT INTO product_allergens VALUES('PA063', 'P020', 'ALG005');
INSERT INTO product_allergens VALUES('PA064', 'P021', 'ALG001');
INSERT INTO product_allergens VALUES('PA065', 'P021', 'ALG002');
INSERT INTO product_allergens VALUES('PA066', 'P021', 'ALG004');
INSERT INTO product_allergens VALUES('PA067', 'P022', 'ALG001');
INSERT INTO product_allergens VALUES('PA068', 'P022', 'ALG002');
INSERT INTO product_allergens VALUES('PA069', 'P022', 'ALG004');
INSERT INTO product_allergens VALUES('PA070', 'P023', 'ALG001');
INSERT INTO product_allergens VALUES('PA071', 'P023', 'ALG002');
INSERT INTO product_allergens VALUES('PA072', 'P023', 'ALG004');
INSERT INTO product_allergens VALUES('PA073', 'P024', 'ALG001');
INSERT INTO product_allergens VALUES('PA074', 'P024', 'ALG002');
INSERT INTO product_allergens VALUES('PA075', 'P024', 'ALG004');
INSERT INTO product_allergens VALUES('PA076', 'P025', 'ALG001');
INSERT INTO product_allergens VALUES('PA077', 'P025', 'ALG002');
INSERT INTO product_allergens VALUES('PA078', 'P025', 'ALG004');
INSERT INTO product_allergens VALUES('PA079', 'P026', 'ALG001');
INSERT INTO product_allergens VALUES('PA080', 'P026', 'ALG002');
INSERT INTO product_allergens VALUES('PA081', 'P026', 'ALG004');
INSERT INTO product_allergens VALUES('PA082', 'P027', 'ALG001');
INSERT INTO product_allergens VALUES('PA083', 'P027', 'ALG002');
INSERT INTO product_allergens VALUES('PA084', 'P027', 'ALG003');
INSERT INTO product_allergens VALUES('PA085', 'P027', 'ALG004');
INSERT INTO product_allergens VALUES('PA086', 'P028', 'ALG001');
INSERT INTO product_allergens VALUES('PA087', 'P028', 'ALG002');
INSERT INTO product_allergens VALUES('PA088', 'P028', 'ALG004');
INSERT INTO product_allergens VALUES('PA089', 'P029', 'ALG001');
INSERT INTO product_allergens VALUES('PA090', 'P029', 'ALG002');
INSERT INTO product_allergens VALUES('PA091', 'P029', 'ALG003');
INSERT INTO product_allergens VALUES('PA092', 'P029', 'ALG004');
INSERT INTO product_allergens VALUES('PA093', 'P030', 'ALG001');
INSERT INTO product_allergens VALUES('PA094', 'P030', 'ALG002');
INSERT INTO product_allergens VALUES('PA095', 'P030', 'ALG003');
INSERT INTO product_allergens VALUES('PA096', 'P030', 'ALG004');
"""

# 재고 샘플 데이터 (각 매장별 상품 재고)
sample_inventory = """
INSERT INTO inventory VALUES('INV001', 'S001', 'P001', 45, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV002', 'S001', 'P002', 32, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV003', 'S001', 'P003', 28, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV004', 'S001', 'P004', 50, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV005', 'S001', 'P005', 38, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV006', 'S001', 'P006', 35, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV007', 'S001', 'P007', 42, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV008', 'S001', 'P008', 39, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV009', 'S001', 'P009', 25, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV010', 'S001', 'P010', 12, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV011', 'S001', 'P011', 10, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV012', 'S001', 'P012', 8, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV013', 'S001', 'P013', 48, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV014', 'S001', 'P014', 44, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV015', 'S001', 'P015', 41, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV016', 'S001', 'P016', 52, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV017', 'S001', 'P017', 49, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV018', 'S001', 'P018', 15, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV019', 'S001', 'P019', 36, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV020', 'S001', 'P020', 22, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV021', 'S001', 'P021', 11, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV022', 'S001', 'P022', 58, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV023', 'S001', 'P023', 29, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV024', 'S001', 'P024', 33, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV025', 'S001', 'P025', 27, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV026', 'S001', 'P026', 9, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV027', 'S001', 'P027', 47, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV028', 'S001', 'P028', 40, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV029', 'S001', 'P029', 5, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV030', 'S001', 'P030', 55, '2024-03-31 10:30:00');
INSERT INTO inventory VALUES('INV031', 'S002', 'P001', 38, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV032', 'S002', 'P002', 29, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV033', 'S002', 'P003', 31, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV034', 'S002', 'P004', 47, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV035', 'S002', 'P005', 35, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV036', 'S002', 'P006', 39, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV037', 'S002', 'P007', 44, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV038', 'S002', 'P008', 36, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV039', 'S002', 'P009', 22, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV040', 'S002', 'P010', 14, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV041', 'S002', 'P011', 0, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV042', 'S002', 'P012', 6, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV043', 'S002', 'P013', 46, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV044', 'S002', 'P014', 41, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV045', 'S002', 'P015', 38, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV046', 'S002', 'P016', 50, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV047', 'S002', 'P017', 46, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV048', 'S002', 'P018', 12, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV049', 'S002', 'P019', 34, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV050', 'S002', 'P020', 19, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV051', 'S002', 'P021', 13, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV052', 'S002', 'P022', 56, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV053', 'S002', 'P023', 27, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV054', 'S002', 'P024', 31, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV055', 'S002', 'P025', 25, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV056', 'S002', 'P026', 7, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV057', 'S002', 'P027', 45, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV058', 'S002', 'P028', 38, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV059', 'S002', 'P029', 3, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV060', 'S002', 'P030', 53, '2024-03-31 11:15:00');
INSERT INTO inventory VALUES('INV061', 'S003', 'P001', 41, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV062', 'S003', 'P002', 35, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV063', 'S003', 'P003', 26, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV064', 'S003', 'P004', 49, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV065', 'S003', 'P005', 37, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV066', 'S003', 'P006', 33, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV067', 'S003', 'P007', 43, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV068', 'S003', 'P008', 37, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV069', 'S003', 'P009', 28, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV070', 'S003', 'P010', 11, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV071', 'S003', 'P011', 9, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV072', 'S003', 'P012', 6, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV073', 'S003', 'P013', 47, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV074', 'S003', 'P014', 43, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV075', 'S003', 'P015', 39, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV076', 'S003', 'P016', 51, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV077', 'S003', 'P017', 48, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV078', 'S003', 'P018', 14, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV079', 'S003', 'P019', 35, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV080', 'S003', 'P020', 21, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV081', 'S003', 'P021', 12, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV082', 'S003', 'P022', 54, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV083', 'S003', 'P023', 30, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV084', 'S003', 'P024', 32, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV085', 'S003', 'P025', 26, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV086', 'S003', 'P026', 8, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV087', 'S003', 'P027', 46, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV088', 'S003', 'P028', 39, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV089', 'S003', 'P029', 4, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV090', 'S003', 'P030', 52, '2024-03-31 09:45:00');
INSERT INTO inventory VALUES('INV091', 'S004', 'P001', 39, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV092', 'S004', 'P002', 30, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV093', 'S004', 'P003', 29, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV094', 'S004', 'P004', 46, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV095', 'S004', 'P005', 36, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV096', 'S004', 'P006', 37, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV097', 'S004', 'P007', 41, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV098', 'S004', 'P008', 38, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV099', 'S004', 'P009', 24, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV100', 'S004', 'P010', 13, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV101', 'S004', 'P011', 8, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV102', 'S004', 'P012', 7, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV103', 'S004', 'P013', 45, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV104', 'S004', 'P014', 42, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV105', 'S004', 'P015', 40, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV106', 'S004', 'P016', 49, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV107', 'S004', 'P017', 47, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV108', 'S004', 'P018', 13, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV109', 'S004', 'P019', 32, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV110', 'S004', 'P020', 20, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV111', 'S004', 'P021', 14, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV112', 'S004', 'P022', 57, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV113', 'S004', 'P023', 28, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV114', 'S004', 'P024', 30, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV115', 'S004', 'P025', 24, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV116', 'S004', 'P026', 10, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV117', 'S004', 'P027', 44, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV118', 'S004', 'P028', 41, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV119', 'S004', 'P029', 2, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV120', 'S004', 'P030', 51, '2024-03-31 12:00:00');
INSERT INTO inventory VALUES('INV121', 'S005', 'P001', 43, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV122', 'S005', 'P002', 33, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV123', 'S005', 'P003', 27, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV124', 'S005', 'P004', 48, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV125', 'S005', 'P005', 38, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV126', 'S005', 'P006', 36, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV127', 'S005', 'P007', 42, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV128', 'S005', 'P008', 39, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV129', 'S005', 'P009', 26, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV130', 'S005', 'P010', 12, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV131', 'S005', 'P011', 11, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV132', 'S005', 'P012', 5, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV133', 'S005', 'P013', 49, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV134', 'S005', 'P014', 44, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV135', 'S005', 'P015', 41, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV136', 'S005', 'P016', 52, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV137', 'S005', 'P017', 50, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV138', 'S005', 'P018', 16, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV139', 'S005', 'P019', 37, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV140', 'S005', 'P020', 23, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV141', 'S005', 'P021', 10, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV142', 'S005', 'P022', 59, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV143', 'S005', 'P023', 31, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV144', 'S005', 'P024', 34, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV145', 'S005', 'P025', 28, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV146', 'S005', 'P026', 9, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV147', 'S005', 'P027', 48, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV148', 'S005', 'P028', 40, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV149', 'S005', 'P029', 6, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV150', 'S005', 'P030', 54, '2024-03-31 10:20:00');
INSERT INTO inventory VALUES('INV151', 'S006', 'P001', 40, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV152', 'S006', 'P002', 32, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV153', 'S006', 'P003', 28, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV154', 'S006', 'P004', 47, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV155', 'S006', 'P005', 37, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV156', 'S006', 'P006', 34, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV157', 'S006', 'P007', 40, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV158', 'S006', 'P008', 36, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV159', 'S006', 'P009', 23, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV160', 'S006', 'P010', 11, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV161', 'S006', 'P011', 10, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV162', 'S006', 'P012', 8, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV163', 'S006', 'P013', 46, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV164', 'S006', 'P014', 42, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV165', 'S006', 'P015', 39, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV166', 'S006', 'P016', 50, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV167', 'S006', 'P017', 47, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV168', 'S006', 'P018', 13, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV169', 'S006', 'P019', 33, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV170', 'S006', 'P020', 19, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV171', 'S006', 'P021', 13, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV172', 'S006', 'P022', 55, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV173', 'S006', 'P023', 29, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV174', 'S006', 'P024', 31, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV175', 'S006', 'P025', 25, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV176', 'S006', 'P026', 7, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV177', 'S006', 'P027', 45, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV178', 'S006', 'P028', 39, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV179', 'S006', 'P029', 3, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV180', 'S006', 'P030', 52, '2024-03-31 08:50:00');
INSERT INTO inventory VALUES('INV181', 'S007', 'P001', 42, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV182', 'S007', 'P002', 34, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV183', 'S007', 'P003', 25, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV184', 'S007', 'P004', 50, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV185', 'S007', 'P005', 39, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV186', 'S007', 'P006', 35, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV187', 'S007', 'P007', 43, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV188', 'S007', 'P008', 40, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV189', 'S007', 'P009', 27, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV190', 'S007', 'P010', 10, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV191', 'S007', 'P011', 12, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV192', 'S007', 'P012', 6, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV193', 'S007', 'P013', 48, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV194', 'S007', 'P014', 45, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV195', 'S007', 'P015', 38, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV196', 'S007', 'P016', 53, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV197', 'S007', 'P017', 49, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV198', 'S007', 'P018', 15, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV199', 'S007', 'P019', 38, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV200', 'S007', 'P020', 24, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV201', 'S007', 'P021', 11, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV202', 'S007', 'P022', 58, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV203', 'S007', 'P023', 32, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV204', 'S007', 'P024', 35, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV205', 'S007', 'P025', 29, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV206', 'S007', 'P026', 8, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV207', 'S007', 'P027', 47, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV208', 'S007', 'P028', 42, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV209', 'S007', 'P029', 5, '2024-03-31 11:40:00');
INSERT INTO inventory VALUES('INV210', 'S007', 'P030', 56, '2024-03-31 11:40:00');
"""

# 프로모션 샘플 데이터
sample_promotions = """
INSERT INTO promotions VALUES('PROMO001', 'P001', 'S001', 10.0, '2024-03-25', '2024-04-07');
INSERT INTO promotions VALUES('PROMO002', 'P003', NULL, 15.0, '2024-03-28', '2024-04-10');
INSERT INTO promotions VALUES('PROMO003', 'P005', 'S002', 12.0, '2024-03-20', '2024-04-05');
INSERT INTO promotions VALUES('PROMO004', 'P010', NULL, 20.0, '2024-03-30', '2024-04-15');
INSERT INTO promotions VALUES('PROMO005', 'P012', 'S003', 18.0, '2024-03-25', '2024-04-08');
INSERT INTO promotions VALUES('PROMO006', 'P007', 'S004', 8.0, '2024-03-29', '2024-04-12');
INSERT INTO promotions VALUES('PROMO007', 'P018', NULL, 25.0, '2024-03-31', '2024-04-20');
INSERT INTO promotions VALUES('PROMO008', 'P020', 'S005', 10.0, '2024-03-26', '2024-04-09');
INSERT INTO promotions VALUES('PROMO009', 'P011', 'S001', 22.0, '2024-03-31', '2024-04-14');
INSERT INTO promotions VALUES('PROMO010', 'P021', 'S006', 15.0, '2024-03-28', '2024-04-11');
INSERT INTO promotions VALUES('PROMO011', 'P029', NULL, 30.0, '2024-03-31', '2024-04-07');
INSERT INTO promotions VALUES('PROMO012', 'P015', 'S007', 12.0, '2024-03-27', '2024-04-10');
"""

# 리뷰 샘플 데이터
sample_reviews = """
INSERT INTO reviews VALUES('R001', 'P001', 5, '정말 맛있는 식빵이에요. 매일 사먹어요!', '2024-03-28');
INSERT INTO reviews VALUES('R002', 'P001', 4, '신선하고 좋아요. 가격이 조금 비싼 편', '2024-03-29');
INSERT INTO reviews VALUES('R003', 'P002', 5, '통곡물 식빵 최고! 건강식으로 추천', '2024-03-27');
INSERT INTO reviews VALUES('R004', 'P002', 4, '맛도 좋고 영양가도 있어요', '2024-03-30');
INSERT INTO reviews VALUES('R005', 'P003', 4, '초코칩 식빵 달콤해서 좋아요', '2024-03-26');
INSERT INTO reviews VALUES('R006', 'P004', 5, '바삭하고 맛있는 롤빵', '2024-03-28');
INSERT INTO reviews VALUES('R007', 'P005', 5, '초콜릿 크림이 정말 부드러워요', '2024-03-29');
INSERT INTO reviews VALUES('R008', 'P006', 4, '딸기가 신선해요. 맛도 좋음', '2024-03-25');
INSERT INTO reviews VALUES('R009', 'P007', 5, '버터 크로와상 정석! 최고의 맛', '2024-03-30');
INSERT INTO reviews VALUES('R010', 'P008', 4, '초콜릿 크로와상도 맛있어요', '2024-03-24');
INSERT INTO reviews VALUES('R011', 'P009', 5, '아몬드가 듬뿍! 고급스러운 맛', '2024-03-28');
INSERT INTO reviews VALUES('R012', 'P010', 5, '생크림 케이크 최고! 케이크는 여기서만', '2024-03-27');
INSERT INTO reviews VALUES('R013', 'P010', 4, '맛있지만 가격이 좀 높아요', '2024-03-30');
INSERT INTO reviews VALUES('R014', 'P011', 5, '초콜릿 가나슈 정말 진한 맛!', '2024-03-29');
INSERT INTO reviews VALUES('R015', 'P012', 4, '딸기 타르트 신선하고 맛있어요', '2024-03-26');
INSERT INTO reviews VALUES('R016', 'P013', 5, '베이글은 여기가 최고!', '2024-03-28');
INSERT INTO reviews VALUES('R017', 'P014', 4, '블루베리 베이글 상큼해요', '2024-03-30');
INSERT INTO reviews VALUES('R018', 'P015', 5, '에브리씬 베이글 건강식으로 좋아요', '2024-03-27');
INSERT INTO reviews VALUES('R019', 'P016', 4, '초콜릿 칩 쿠키 크기도 크고 맛있어요', '2024-03-25');
INSERT INTO reviews VALUES('R020', 'P017', 5, '더블 초콜릿 초콜릿 중독자 필수', '2024-03-29');
INSERT INTO reviews VALUES('R021', 'P018', 5, '마카롱 세트 선물용으로 최고!', '2024-03-28');
INSERT INTO reviews VALUES('R022', 'P019', 4, '소금 버터 카라멜 베이글 중독성 있어요', '2024-03-26');
INSERT INTO reviews VALUES('R023', 'P020', 5, '호두 바나나 빵 견과류 향 좋아요', '2024-03-30');
INSERT INTO reviews VALUES('R024', 'P021', 4, '레몬 파운드 케이크 상큼한 맛 최고', '2024-03-27');
INSERT INTO reviews VALUES('R025', 'P022', 5, '딸기 마카롱 색상도 예쁘고 맛있어요', '2024-03-29');
INSERT INTO reviews VALUES('R026', 'P023', 4, '말차 라떼 롤빵 특별해요', '2024-03-28');
INSERT INTO reviews VALUES('R027', 'P024', 5, '앙금 식빵 팥 향 좋아요', '2024-03-25');
INSERT INTO reviews VALUES('R028', 'P025', 5, '까망베르 치즈 크로와상 고급스러워요', '2024-03-30');
INSERT INTO reviews VALUES('R029', 'P026', 4, '라즈베리 타르트 신맛 좋아요', '2024-03-26');
INSERT INTO reviews VALUES('R030', 'P027', 5, '오트밀 클러스터 쿠키 건강식 최고', '2024-03-28');
"""

In [7]:
# 빵집 SQLite 데이터베이스 생성
def create_bakery_db():
    db_file = os.path.join(tempfile.gettempdir(), "bakery.db")
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()

    # 기존 테이블 삭제 (이미 존재하는 경우)
    # 외래키 제약조건 때문에 자식 테이블부터 삭제
    cursor.executescript("""
    DROP TABLE IF EXISTS reviews;
    DROP TABLE IF EXISTS promotions;
    DROP TABLE IF EXISTS product_allergens;
    DROP TABLE IF EXISTS inventory;
    DROP TABLE IF EXISTS allergens;
    DROP TABLE IF EXISTS store_hours;
    DROP TABLE IF EXISTS products;
    DROP TABLE IF EXISTS categories;
    DROP TABLE IF EXISTS stores;
    """)

    # DDL 및 샘플 데이터 실행
    cursor.executescript(bakery_schema)
    cursor.executescript(sample_stores)
    cursor.executescript(sample_categories)
    cursor.executescript(sample_products)
    cursor.executescript(sample_store_hours)
    cursor.executescript(sample_allergens)
    cursor.executescript(sample_product_allergens)
    cursor.executescript(sample_inventory)
    cursor.executescript(sample_promotions)
    cursor.executescript(sample_reviews)

    conn.commit()
    conn.close()

    # LangChain SQL 데이터베이스 인스턴스 반환
    return SQLDatabase.from_uri(f"sqlite:///{db_file}")

# 데이터베이스 생성 및 확인
db = create_bakery_db()
print("✅ 빵집 데이터베이스 생성 완료!")

# 테이블 정보 확인
print("\n📋 생성된 테이블 목록:")
print(db.get_table_names())

# 샘플 쿼리로 데이터 확인
print("\n📊 상품 데이터 샘플:")
result = db.run("SELECT product_id, product_name, price FROM products LIMIT 5;")
print(result)

print("\n📊 매장 데이터 샘플:")
result = db.run("SELECT store_id, store_name, city FROM stores LIMIT 5;")
print(result)

print("\n📊 재고 데이터 샘플:")
result = db.run("SELECT i.store_id, p.product_name, i.quantity_in_stock FROM inventory i JOIN products p ON i.product_id = p.product_id LIMIT 5;")
print(result)

✅ 빵집 데이터베이스 생성 완료!

📋 생성된 테이블 목록:
['allergens', 'categories', 'inventory', 'product_allergens', 'products', 'promotions', 'reviews', 'store_hours', 'stores']

📊 상품 데이터 샘플:
[('P001', '프리미엄 버터 식빵', 5500), ('P002', '통곡물 건강식빵', 6000), ('P003', '초코칩 식빵', 5800), ('P004', '슈가 롤빵', 3500), ('P005', '초콜릿 크림 롤빵', 4200)]

📊 매장 데이터 샘플:
[('S001', '서울 강남점', '서울'), ('S002', '서울 명동점', '서울'), ('S003', '부산 해운대점', '부산'), ('S004', '대구 중앙점', '대구'), ('S005', '인천 연수점', '인천')]

📊 재고 데이터 샘플:
[('S001', '프리미엄 버터 식빵', 45), ('S001', '통곡물 건강식빵', 32), ('S001', '초코칩 식빵', 28), ('S001', '슈가 롤빵', 50), ('S001', '초콜릿 크림 롤빵', 38)]


/tmp/ipykernel_50228/1602457518.py:45: LangChainDeprecationWarning: The method `SQLDatabase.get_table_names` was deprecated in langchain-community 0.0.1 and will be removed in 1.0. Use `get_usable_table_names` instead.
  print(db.get_table_names())


### 합성 데이터 기반으로 조회가 가능한 정보 종류 리스트입니다.
#### 실제 상황을 반영하기 위해, 1~7번은 미리 Tool로 정의하고 나머지는 Text-to-SQL로 구현합니다.

[Tool - Fixed SQL]
1. 특정 매장의 특정 상품 재고량
2. 특정 도시의 모든 매장 정보
3. 특정 카테고리의 모든 상품 목록
4. 특정 상품의 가격 및 설명
5. 특정 매장의 운영시간
6. 특정 상품을 취급하는 모든 매장
7. 현재 할인 중인 상품 목록

[Text-to-SQL]

8. 특정 상품의 알레르겐 정보
9. 특정 알레르겐이 없는 상품 검색
10. 특정 상품의 평균 평점 및 리뷰
11. 5천원대 상품 검색
12. 재고가 부족한 상품(특정 매장)
13. 모든 매장별 특정 상품 재고 비교

In [33]:
import ast

query = f"""
  SELECT i.quantity_in_stock
    FROM inventory i
    JOIN products p
    JOIN stores s
    ON i.product_id = p.product_id and i.store_id = s.store_id
    WHERE s.store_name LIKE '%강남점%' AND p.product_name LIKE '%생크림 케이크%'
  """

rows = db.run(query)

print(rows)
if isinstance(rows, str):
    s = ast.literal_eval(rows)
    print(s)
    print(type(s))

[(12,)]
[(12,)]
<class 'list'>


In [9]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
import pandas as pd
import json
import ast

# ============================================================================
# 1. SQL 실행 헬퍼 함수 추가
# ============================================================================

def execute_sql_query(query: str) -> list:
    """
    SQL 쿼리를 실행하고 튜플 리스트로 변환합니다.

    db.run()은 문자열 형태의 리스트를 반환하므로 ast.literal_eval()로 변환합니다.
    예시:
        문자열 입력: "[('서울 강남점', '생크림 케이크', 12, 35000)]"
        리스트 출력: [('서울 강남점', '생크림 케이크', 12, 35000)]

    Returns:
        튜플 리스트. 쿼리 실패 시 빈 리스트 []
    """
    try:
        result = db.run(query)

        # 문자열을 안전하게 파이썬 객체로 변환
        parsed = ast.literal_eval(result)

        # 리스트인지 확인
        if isinstance(parsed, list):
            return parsed
        else:
            return []

    except (ValueError, SyntaxError):
        # ast.literal_eval 실패 (문자열이 유효한 파이썬 리터럴이 아님)
        return []
    except Exception as e:
        # 기타 에러 (쿼리 실패 등)
        return []


def escape_sql_string(value: str) -> str:
    """
    SQL 문자열 값을 이스케이프합니다.
    """
    return value.replace("'", "''")


# ============================================================================
# 2. Fixed SQL Tools (간소화된 버전)
# ============================================================================

@tool
def search_store_product_stock(store_name: str, product_name: str) -> str:
    """
    특정 매장의 특정 빵 제품 재고 수량을 조회합니다.

    Args:
        store_name: 매장 이름 (예: '서울 강남점', '부산 해운대점')
        product_name: 빵 제품명 (예: '버터 식빵', '초콜릿 케이크')

    Returns:
        매장명, 제품명, 현재 재고량을 포함한 결과
    """
    escaped_store = escape_sql_string(store_name)
    escaped_product = escape_sql_string(product_name)

    query = f"""
    SELECT s.store_name, p.product_name, i.quantity_in_stock, p.price
    FROM inventory i
    JOIN products p ON i.product_id = p.product_id
    JOIN stores s ON i.store_id = s.store_id
    WHERE s.store_name LIKE '%{escaped_store}%' AND p.product_name LIKE '%{escaped_product}%'
    LIMIT 1
    """

    try:
        rows = execute_sql_query(query)

        if not rows:
            return json.dumps({
                "status": "not_found",
                "message": f"'{store_name}'에서 '{product_name}'을(를) 찾을 수 없습니다.",
                "data": None
            }, ensure_ascii=False)

        row = rows[0]

        return json.dumps({
            "status": "success",
            "data": {
                "store_name": row[0],
                "product_name": row[1],
                "quantity": row[2],
                "price": row[3]
            }
        }, ensure_ascii=False)

    except Exception as e:
        return json.dumps({
            "status": "error",
            "message": f"조회 중 오류가 발생했습니다: {str(e)}",
            "data": None
        }, ensure_ascii=False)


@tool
def search_stores_in_city(city_name: str) -> str:
    """
    특정 도시에 있는 모든 빵집 매장을 조회합니다.

    Args:
        city_name: 도시명 (예: '서울', '부산', '대구', '인천', '대전', '광주')

    Returns:
        해당 도시의 모든 매장 정보 (이름, 주소, 전화번호, 개점일자)
    """
    escaped_city = escape_sql_string(city_name)

    query = f"""
    SELECT store_name, address, phone, opened_date
    FROM stores
    WHERE city = '{escaped_city}'
    ORDER BY store_name
    """

    try:
        rows = execute_sql_query(query)

        if not rows:
            return json.dumps({
                "status": "not_found",
                "message": f"'{city_name}'에 등록된 매장이 없습니다.",
                "data": None
            }, ensure_ascii=False)

        stores_data = []
        for row in rows:
            stores_data.append({
                "name": row[0],
                "address": row[1],
                "phone": row[2],
                "opened_date": row[3]
            })

        return json.dumps({
            "status": "success",
            "data": stores_data
        }, ensure_ascii=False)

    except Exception as e:
        return json.dumps({
            "status": "error",
            "message": f"조회 중 오류가 발생했습니다: {str(e)}",
            "data": None
        }, ensure_ascii=False)


@tool
def search_products_by_category(category_name: str) -> str:
    """
    특정 카테고리의 모든 빵 제품을 조회합니다.

    Args:
        category_name: 카테고리명 (예: '식빵', '롤빵', '크로와상', '케이크', '베이글', '쿠키/마카롱')

    Returns:
        해당 카테고리의 모든 상품 (상품명, 가격, 설명)
    """
    escaped_category = escape_sql_string(category_name)

    query = f"""
    SELECT p.product_name, p.price, p.description
    FROM products p
    JOIN categories c ON p.category_id = c.category_id
    WHERE c.category_name = '{escaped_category}'
    ORDER BY p.price
    """

    try:
        rows = execute_sql_query(query)

        if not rows:
            return json.dumps({
                "status": "not_found",
                "message": f"'{category_name}' 카테고리에 상품이 없습니다.",
                "data": None
            }, ensure_ascii=False)

        products_data = []
        for row in rows:
            products_data.append({
                "name": row[0],
                "price": row[1],
                "description": row[2]
            })

        return json.dumps({
            "status": "success",
            "data": products_data
        }, ensure_ascii=False)

    except Exception as e:
        return json.dumps({
            "status": "error",
            "message": f"조회 중 오류가 발생했습니다: {str(e)}",
            "data": None
        }, ensure_ascii=False)


@tool
def search_product_details(product_name: str) -> str:
    """
    특정 빵 제품의 상세 정보 (가격, 설명, 카테고리)를 조회합니다.

    Args:
        product_name: 제품명 (예: '버터 식빵', '생크림 케이크', '베이글')

    Returns:
        제품명, 가격, 설명, 카테고리 정보
    """
    escaped_product = escape_sql_string(product_name)

    query = f"""
    SELECT p.product_name, p.price, p.description, c.category_name
    FROM products p
    JOIN categories c ON p.category_id = c.category_id
    WHERE p.product_name LIKE '%{escaped_product}%'
    LIMIT 1
    """

    try:
        rows = execute_sql_query(query)

        if not rows:
            return json.dumps({
                "status": "not_found",
                "message": f"'{product_name}' 상품을 찾을 수 없습니다.",
                "data": None
            }, ensure_ascii=False)

        row = rows[0]

        return json.dumps({
            "status": "success",
            "data": {
                "name": row[0],
                "price": row[1],
                "description": row[2],
                "category": row[3]
            }
        }, ensure_ascii=False)

    except Exception as e:
        return json.dumps({
            "status": "error",
            "message": f"조회 중 오류가 발생했습니다: {str(e)}",
            "data": None
        }, ensure_ascii=False)


@tool
def search_store_operating_hours(store_name: str) -> str:
    """
    특정 매장의 요일별 운영시간을 조회합니다.

    Args:
        store_name: 매장명 (예: '서울 강남점', '부산 해운대점')

    Returns:
        월~일의 오픈시간과 마감시간
    """
    escaped_store = escape_sql_string(store_name)

    query = f"""
    SELECT day_of_week, opening_time, closing_time
    FROM store_hours sh
    JOIN stores s ON sh.store_id = s.store_id
    WHERE s.store_name LIKE '%{escaped_store}%'
    ORDER BY
        CASE WHEN day_of_week='월' THEN 1
             WHEN day_of_week='화' THEN 2
             WHEN day_of_week='수' THEN 3
             WHEN day_of_week='목' THEN 4
             WHEN day_of_week='금' THEN 5
             WHEN day_of_week='토' THEN 6
             WHEN day_of_week='일' THEN 7
        END
    """

    try:
        rows = execute_sql_query(query)

        if not rows:
            return json.dumps({
                "status": "not_found",
                "message": f"'{store_name}'의 운영시간 정보를 찾을 수 없습니다.",
                "data": None
            }, ensure_ascii=False)

        hours_data = []
        for row in rows:
            hours_data.append({
                "day": row[0],
                "open_time": row[1],
                "close_time": row[2]
            })

        return json.dumps({
            "status": "success",
            "data": hours_data
        }, ensure_ascii=False)

    except Exception as e:
        return json.dumps({
            "status": "error",
            "message": f"조회 중 오류가 발생했습니다: {str(e)}",
            "data": None
        }, ensure_ascii=False)


@tool
def search_stores_selling_product(product_name: str) -> str:
    """
    특정 빵 제품을 판매하는 모든 매장을 조회합니다.

    Args:
        product_name: 제품명 (예: '생크림 케이크', '버터 크로와상')

    Returns:
        해당 상품을 판매하는 매장 목록 (재고 수량 포함)
    """
    escaped_product = escape_sql_string(product_name)

    query = f"""
    SELECT DISTINCT s.store_name, s.city, s.address, i.quantity_in_stock
    FROM inventory i
    JOIN stores s ON i.store_id = s.store_id
    JOIN products p ON i.product_id = p.product_id
    WHERE p.product_name LIKE '%{escaped_product}%' AND i.quantity_in_stock > 0
    ORDER BY s.city, s.store_name
    """

    try:
        rows = execute_sql_query(query)

        if not rows:
            return json.dumps({
                "status": "not_found",
                "message": f"'{product_name}'의 재고가 있는 매장이 없습니다.",
                "data": None
            }, ensure_ascii=False)

        stores_data = []
        for row in rows:
            stores_data.append({
                "name": row[0],
                "city": row[1],
                "address": row[2],
                "stock": row[3]
            })

        return json.dumps({
            "status": "success",
            "data": stores_data
        }, ensure_ascii=False)

    except Exception as e:
        return json.dumps({
            "status": "error",
            "message": f"조회 중 오류가 발생했습니다: {str(e)}",
            "data": None
        }, ensure_ascii=False)


@tool
def search_current_promotions() -> str:
    """
    현재 진행 중인 모든 할인 이벤트를 조회합니다.

    Returns:
        할인 중인 제품명, 할인율, 적용 매장, 할인 종료일
    """
    query = """
    SELECT p.product_name, pr.discount_percent, pr.valid_until,
           CASE WHEN pr.store_id IS NULL THEN '전체 매장'
                ELSE (SELECT store_name FROM stores WHERE store_id = pr.store_id)
           END as store_info
    FROM promotions pr
    JOIN products p ON pr.product_id = p.product_id
    WHERE DATE(pr.valid_until) >= DATE('now')
    ORDER BY pr.discount_percent DESC, pr.valid_until ASC
    """

    try:
        rows = execute_sql_query(query)

        if not rows:
            return json.dumps({
                "status": "not_found",
                "message": "현재 진행 중인 할인 이벤트가 없습니다.",
                "data": None
            }, ensure_ascii=False)

        promotions_data = []
        for row in rows:
            promotions_data.append({
                "product": row[0],
                "discount_percent": row[1],
                "until_date": row[2],
                "store": row[3]
            })

        return json.dumps({
            "status": "success",
            "data": promotions_data
        }, ensure_ascii=False)

    except Exception as e:
        return json.dumps({
            "status": "error",
            "message": f"조회 중 오류가 발생했습니다: {str(e)}",
            "data": None
        }, ensure_ascii=False)


# ============================================================================
# 3. Tools List 생성
# ============================================================================

tools = [
    search_store_product_stock,
    search_stores_in_city,
    search_products_by_category,
    search_product_details,
    search_store_operating_hours,
    search_stores_selling_product,
    search_current_promotions,
]


# ============================================================================
# 4. LLM에 Tools 바인딩
# ============================================================================

from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI
import sqlite3
import os

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    api_key=os.environ.get("OPENAI_API_KEY")
)

llm_with_tools = llm.bind_tools(tools)

In [10]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages
from pydantic import BaseModel

class QueryResult(BaseModel):
    """쿼리 실행 결과의 통일된 형식"""
    query_type: str  # "fixed_tool" 또는 "text_to_sql"
    tool_name: str = ""  # fixed_tool의 경우 tool 이름
    sql_query: str = ""  # text_to_sql의 경우 생성된 SQL
    raw_result: str  # 데이터베이스 조회 결과
    user_question: str  # 원래 사용자 질문

class AgentState(TypedDict):
    """에이전트의 상태를 관리하는 Schema"""
    messages: Annotated[list[AnyMessage], add_messages]
    query: str
    current_tool: str = ""
    tool_result: str = ""
    sql_result: str = ""
    final_response: str = ""
    query_result: QueryResult | None

#### 1~7번 tool 중, call할 tool을 LLM이 결정합니다.

In [12]:
from langgraph.types import Command

def fixed_sql_tools_node(state: AgentState) -> Command[AgentState]:
    """
    Fixed SQL Tools를 사용할 LLM 의사결정 노드
    """

    system_prompt = """당신은 빵집 고객 응답 AI입니다.
    사용자의 질문에 대해 다음 7가지 도구 중 적절한 도구를 선택하세요:

    1. search_store_product_stock - 특정 매장의 특정 상품 재고 조회
      예: "강남점에 식빵 남았나요?"

    2. search_stores_in_city - 특정 도시의 모든 매장 조회
      예: "서울에 몇 개 지점이 있어요?"

    3. search_products_by_category - 특정 카테고리의 모든 상품 조회
      예: "롤빵 종류가 뭐가 있어요?"

    4. search_product_details - 특정 상품의 상세 정보 조회
      예: "식빵 가격이 얼마예요?"

    5. search_store_operating_hours - 매장 운영시간 조회
      예: "강남점 영업시간이 어떻게 되나요?"

    6. search_stores_selling_product - 특정 상품을 파는 모든 매장 조회
      예: "생크림 케이크는 어디서 팔아요?"

    7. search_current_promotions - 현재 진행 중인 할인 조회
      예: "지금 할인하는 빵이 뭐가 있어요?"

    사용자의 질문에 완벽히 답변하기 위해 필요하다면 여러 개의 도구를 한 번에 선택하세요
"""

    messages = state['messages']

    response = llm_with_tools.invoke(
        [{"role": "system", "content": system_prompt}] + messages
    )

    updated_messages = messages + [response]

    return Command(
        update={"messages": updated_messages}
    )

#### LLM이 선택한 tool을 실행합니다.

In [13]:
from langgraph.types import Command

def tools_node(state: AgentState) -> Command[AgentState]:
    """
    LLM이 호출한 tool을 실행하고 결과를 구조화된 형식으로 반환합니다.
    """

    messages = state['messages']
    last_message = messages[-1]

    if not hasattr(last_message, 'tool_calls') or not last_message.tool_calls:
        return Command(update={"messages": messages})

    tool_results = []

    # Tool 호출 정보 추출
    for tool_call in last_message.tool_calls:
      tool_name = tool_call["name"]
      tool_input = tool_call["args"]
      tool_call_id = tool_call["id"]

      # 해당 tool 찾기
      selected_tool = None
      for tool in tools:
          if tool.name == tool_name:
              selected_tool = tool
              break

      if not selected_tool:
          continue

      # Tool 실행
      tool_result = selected_tool.invoke(tool_input)

      # 결과를 QueryResult로 변환
      try:
          result_data = json.loads(tool_result)
          query_result = QueryResult(
              query_type="fixed_tool",
              tool_name=tool_name,
              raw_result=tool_result,
              user_question=state['query']
          )
      except:
          query_result = QueryResult(
              query_type="fixed_tool",
              tool_name=tool_name,
              raw_result=tool_result,
              user_question=state['query']
          )

      # 메시지에 tool result 추가
      tool_message = ToolMessage(
          content=tool_result,
          tool_call_id=tool_call_id,
          name=tool_name
      )

      tool_results.append({
            "tool_message": tool_message,
            "query_result": query_result
        })

    # 모든 tool_messages를 messages에 추가
    updated_messages = messages + [result["tool_message"] for result in tool_results]

    # 마지막 query_result만 state에 저장 (또는 모두 누적)
    final_query_result = tool_results[-1]["query_result"] if tool_results else None

    return Command(
        update={
            "messages": updated_messages,
            "query_result": final_query_result
        }
    )





#### 사용자의 쿼리와 DB Schema를 기반으로 SQL을 생성하고 실행합니다.

In [14]:
def text_to_sql_node(state: AgentState) -> Command[AgentState]:
    """
    복잡한 사용자 질문을 SQL로 생성하고 실행하는 노드
    구조화된 QueryResult 형식으로 반환합니다.
    """

    db_schema_info = """
    데이터베이스 스키마:

    1. stores (store_id, store_name, address, city, phone, opened_date)
    2. categories (category_id, category_name, description)
    3. products (product_id, product_name, category_id, price, description, image_url)
    4. store_hours (store_hours_id, store_id, day_of_week, opening_time, closing_time)
    5. inventory (inventory_id, store_id, product_id, quantity_in_stock, last_updated)
    6. allergens (allergen_id, allergen_name)
    7. product_allergens (product_allergen_id, product_id, allergen_id)
    8. promotions (promotion_id, product_id, store_id, discount_percent, valid_from, valid_until)
    9. reviews (review_id, product_id, rating, comment, review_date)
    """

    few_shot_examples = """
    [예시 1] "글루텐 없는 빵이 있어요?"
    SQL:
    SELECT DISTINCT p.product_name, p.price, c.category_name
    FROM products p
    JOIN categories c ON p.category_id = c.category_id
    WHERE p.description LIKE '%글루텐%' OR p.product_name LIKE '%글루텐%'
    ORDER BY p.price;

    [예시 2] "딸기 제품 중에서 가장 비싼 것은?"
    SQL:
    SELECT product_name, price
    FROM products
    WHERE product_name LIKE '%딸기%'
    ORDER BY price DESC
    LIMIT 1;

    [예시 3] "평점 4점 이상인 상품들"
    SQL:
    SELECT p.product_name, AVG(r.rating) as avg_rating, COUNT(r.review_id) as review_count
    FROM products p
    LEFT JOIN reviews r ON p.product_id = r.product_id
    GROUP BY p.product_id, p.product_name
    HAVING AVG(r.rating) >= 4
    ORDER BY avg_rating DESC;

    [예시 4] "계란 없는 빵 종류는?"
    SQL:
    SELECT DISTINCT p.product_name, p.price, c.category_name
    FROM products p
    JOIN categories c ON p.category_id = c.category_id
    WHERE p.product_id NOT IN (
        SELECT pa.product_id
        FROM product_allergens pa
        WHERE pa.allergen_id = (SELECT allergen_id FROM allergens WHERE allergen_name = '계란')
    )
    ORDER BY p.price;
    """

    system_prompt = f"""당신은 SQLite 전문가입니다.

사용자의 자연어 질문을 SQL SELECT 쿼리로 변환하세요.
SQL은 정확하고 실행 가능해야 합니다.

{db_schema_info}

{few_shot_examples}

중요 사항:
- 오직 SELECT 문만 생성하세요 (UPDATE, DELETE 금지)
- SQL 문법을 정확하게 작성하세요
- 필요한 경우 JOIN, GROUP BY, HAVING 등을 사용하세요
- 결과는 읽기 좋도록 정렬하세요
- SQL만 반환하세요 (설명 불필요)
"""

    user_question = state['query']

    # LLM에 SQL 생성 요청
    sql_generation_response = llm.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"질문: {user_question}"}
        ]
    )

    generated_sql = sql_generation_response.content.strip()

    # SQL에서 마크다운 코드 블록 제거
    if "```sql" in generated_sql:
        generated_sql = generated_sql.split("```sql")[1].split("```")[0].strip()
    elif "```" in generated_sql:
        generated_sql = generated_sql.split("```")[1].split("```")[0].strip()

    # SQL 실행
    sql_result = ""
    try:
        sql_result = db.run(generated_sql)

        if "Error" in sql_result:
            # SQL 재생성 시도
            retry_prompt = f"""이전 SQL에서 오류가 발생했습니다:

오류: {sql_result}

생성한 SQL: {generated_sql}

사용자 질문: {user_question}

오류를 수정하고 새로운 SQL을 생성하세요."""

            retry_response = llm.invoke(
                [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": retry_prompt}
                ]
            )

            retry_sql = retry_response.content.strip()
            if "```sql" in retry_sql:
                retry_sql = retry_sql.split("```sql")[1].split("```")[0].strip()
            elif "```" in retry_sql:
                retry_sql = retry_sql.split("```")[1].split("```")[0].strip()

            sql_result = db.run(retry_sql)
            generated_sql = retry_sql

        if not sql_result.strip():
            sql_result = "조건에 맞는 결과가 없습니다."

    except Exception as e:
        sql_result = f"데이터베이스 오류: {str(e)}"

    # QueryResult로 변환
    query_result = QueryResult(
        query_type="text_to_sql",
        sql_query=generated_sql,
        raw_result=sql_result,
        user_question=user_question
    )

    # 메시지에 추가
    messages = state['messages']
    result_message = AIMessage(
        content=f"SQL 쿼리를 실행했습니다:\n```sql\n{generated_sql}\n```"
    )

    return Command(
        update={
            "messages": messages + [result_message],
            "query_result": query_result
        }
    )


#### tool call or text-to-sql의 결과를 파싱하고 사용자에게 응답하기 위한 자연어를 생성합니다.

In [15]:
def format_response_node(state: AgentState) -> Command[AgentState]:
    """
    구조화된 QueryResult를 자연어로 변환하는 최종 노드
    Fixed Tool 결과와 Text-to-SQL 결과를 동일하게 처리합니다.
    """

    if not state['query_result']:
        return Command(update={"final_response": "응답을 생성할 수 없습니다."})

    query_result = state['query_result']
    user_question = state['query']

    # QueryResult에서 정보 추출
    if query_result.query_type == "fixed_tool":
        context = f"""
사용자 질문: {user_question}
사용된 도구: {query_result.tool_name}
조회 결과: {query_result.raw_result}
"""
    else:  # text_to_sql
        context = f"""
사용자 질문: {user_question}
실행된 SQL: {query_result.sql_query}
조회 결과: {query_result.raw_result}
"""

    # LLM으로 자연어 변환
    conversion_prompt = f"""다음 데이터베이스 조회 결과를 사용자 질문에 대한 친절하고 읽기 쉬운 자연어 응답으로 변환하세요.

{context}

응답 작성 가이드:
1. 조회 결과를 명확하게 설명하세요
2. 이모지를 적절히 사용하여 가독성을 높이세요
3. 데이터가 없으면 그 사실을 친절하게 알려주세요
4. 결과 데이터를 구조화되어 읽기 쉽게 포맷하세요
5. 원문 데이터가 "not_found" 또는 "error" 상태인 경우 그에 맞게 응답하세요

친절하고 자연스러운 한국어 응답만 제공하세요."""

    conversion_response = llm.invoke(
        [{"role": "user", "content": conversion_prompt}]
    )

    final_response = conversion_response.content

    return Command(update={"final_response": final_response})

#### 사용자의 질문 의도를 파악하고 tool 호출 or Text-to-SQL 경로를 결정합니다.
#### 한번에 다양한 tool의 name, description을 컨텍스트로 주입 후, tool call or text-to-sql을 결정하면 환각 현상이 발생할 수 있어
#### 별도의 intent node로 분리합니다.

In [89]:
def classify_intent_node(state: AgentState) -> Command[AgentState]:

    user_query = state['query']

    classification_prompt = """사용자의 질문을 분석하여 다음 중 하나로 분류하세요:

    1. FIXED_TOOLS - 다음 7가지 도구로 처리 가능:
      • 특정 매장의 특정 상품 재고 조회
      • 특정 도시의 모든 매장 조회
      • 특정 카테고리의 모든 상품 조회
      • 특정 상품 상세 정보 조회
      • 매장 운영시간 조회
      • 특정 상품을 파는 모든 매장 조회
      • 현재 진행 중인 할인 조회

    2. TEXT_TO_SQL - 복잡한 쿼리가 필요:
      • 위 FIXED_TOOLS로 결과 도출이 어려운 정보 조회

    예시1)
    질문 : 강남점 전화번호랑 월요일 운영시간 알려줘
    분석 결과 : 강남점 전화번호는 '특정 도시의 모든 매장 조회' 도구로 조회 가능하고, 운영 시간은 '매장 운영시간 조회'도구로 조회 가능
    분류 결과 : FIXED_TOOLS

    예시2)
    질문 : 대구점에서 판매하는 빵 종류랑 상품별 리뷰 정보 알려줘
    분석 결과 : 대구점에서 판매하는 빵 종류는 '특정 매장의 특정 상품 재고 조회' 도구로 조회 가능하지만, 상품별 리뷰 정보는 도구로 조회 불가능
    분류 결과 : TEXT_TO_SQL

    사용자 질문: {user_query}

    응답 형식: {{"intent": "FIXED_TOOLS" 또는 "TEXT_TO_SQL"}}"""

    intent_response = llm.invoke(
        [{"role": "user", "content": classification_prompt}]
    )

    intent_text = intent_response.content.strip()

    # JSON 파싱
    try:
        if "{" in intent_text and "}" in intent_text:
            intent_json = json.loads(intent_text[intent_text.index("{"):intent_text.index("}")+1])
            intent = intent_json.get("intent", "TEXT_TO_SQL")
        else:
            if "FIXED_TOOLS" in intent_text:
                intent = "FIXED_TOOLS"
            else:
                intent = "TEXT_TO_SQL"
    except:
        intent = "TEXT_TO_SQL"

    return Command(update={"current_tool": intent})

In [90]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver

def route_by_intent(state: AgentState) -> str:
    """Intent에 따라 노드를 라우팅합니다."""
    intent = state['current_tool']
    return "fixed_sql_tools" if intent == "FIXED_TOOLS" else "text_to_sql"


def create_bakery_agent_graph():
    """LangGraph 에이전트 워크플로우"""

    workflow = StateGraph(AgentState)

    # 노드 추가
    workflow.add_node("classify_intent", classify_intent_node)
    workflow.add_node("fixed_sql_tools", fixed_sql_tools_node)
    workflow.add_node("tools_executor", tools_node)
    workflow.add_node("text_to_sql", text_to_sql_node)
    workflow.add_node("format_response", format_response_node)

    # 엣지 정의
    workflow.add_edge(START, "classify_intent")

    # 조건부 라우팅
    workflow.add_conditional_edges(
        "classify_intent",
        route_by_intent,
        {
            "fixed_sql_tools": "fixed_sql_tools",
            "text_to_sql": "text_to_sql"
        }
    )

    # Fixed SQL Tools 경로
    workflow.add_edge("fixed_sql_tools", "tools_executor")
    workflow.add_edge("tools_executor", "format_response")

    # Text-to-SQL 경로
    workflow.add_edge("text_to_sql", "format_response")

    # 최종 응답
    workflow.add_edge("format_response", END)

    memory = InMemorySaver()

    return workflow.compile(checkpointer=memory)

bakery_agent = create_bakery_agent_graph()

def run_bakery_chatbot(user_query: str, config) -> str:
  """
  사용자 질문에 대한 빵집 챗봇 응답을 생성합니다.

  Args:
      user_query: 사용자의 질문

  Returns:
      챗봇의 최종 응답
  """

  initial_state = AgentState(
      messages=[HumanMessage(content=user_query)],
      query=user_query,
      current_tool="",
      query_result=None,
      final_response=""
  )

  result = bakery_agent.invoke(initial_state, config)

  return result




In [91]:
config_1 = {"configurable" : {"thread_id" : "1"}}

In [92]:
test_query_1 = "강남점에 생크림 케이크 몇 개 있어요?"
result_1 = run_bakery_chatbot(test_query_1, config_1)
print(result_1['final_response'])

안녕하세요! 😊

서울 강남점에 있는 생크림 케이크에 대해 알려드릴게요:

- **매장 이름**: 서울 강남점
- **제품 이름**: 생크림 케이크
- **재고 수량**: 12개
- **가격**: 35,000원

필요한 정보가 도움이 되셨길 바랍니다! 더 궁금한 점이 있으면 언제든지 말씀해 주세요. 🎂✨


In [37]:
test_query_2 = "롤빵 종류 뭐뭐있나요?"
result_2 = run_bakery_chatbot(test_query_2, config_1)
print(result_2['final_response'])

롤빵 종류를 찾으셨군요! 😊 아래는 저희가 제공하는 다양한 롤빵 종류입니다:

1. **딸기 크림 롤빵** 🍓 - 4,500원
2. **말차 라떼 롤빵** 🍵 - 4,800원
3. **슈가 롤빵** 🍬 - 3,500원
4. **초콜릿 크림 롤빵** 🍫 - 4,200원

각 롤빵은 독특한 맛과 향을 자랑하니, 취향에 맞게 선택해보세요! 🥰


In [20]:
test_query_3 = "대구 빵집 요일별 오픈시간 알려줘"
result_3 = run_bakery_chatbot(test_query_3, config_1)
print(result_3['final_response'])

대구에 있는 빵집의 요일별 오픈 시간을 알려드릴게요! 😊

**대구 중앙점**의 영업 시간은 다음과 같습니다:

- **월요일**: 오전 6시 30분 - 오후 9시
- **화요일**: 오전 6시 30분 - 오후 9시
- **수요일**: 오전 6시 30분 - 오후 9시
- **목요일**: 오전 6시 30분 - 오후 9시
- **금요일**: 오전 6시 30분 - 오후 9시
- **토요일**: 오전 8시 - 오후 8시
- **일요일**: 오전 9시 - 오후 7시

맛있는 빵을 즐기시길 바라요! 🥐🍞


In [94]:
test_query_4 = "프리미엄 버터 식빵 리뷰 어때?"
result_4 = run_bakery_chatbot(test_query_4, config_1)
print(result_4['final_response'])

프리미엄 버터 식빵에 대한 최근 리뷰를 확인해보았습니다! 🥖✨

1. **2024년 3월 29일** - ⭐️ 4점  
   "신선하고 좋아요. 가격이 조금 비싼 편이에요."

2. **2024년 3월 28일** - ⭐️ 5점  
   "정말 맛있는 식빵이에요. 매일 사먹어요!"

이 식빵은 신선하고 맛있다는 평가를 받고 있으며, 특히 매일 먹고 싶을 정도로 맛있다는 리뷰가 있네요. 다만, 가격이 조금 비싸다는 의견도 있으니 참고하세요! 😊


#### 답변 검증을 위해 DB 데이터를 시각화합니다.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import ast

# ============================================================================
# 데이터 로드
# ============================================================================

stores_data = db.run("SELECT * FROM stores")
products_data = db.run("SELECT * FROM products")
inventory_data = db.run("SELECT * FROM inventory")
categories_data = db.run("SELECT * FROM categories")
store_hours_data = db.run("SELECT * FROM store_hours")
product_allergens_data = db.run("SELECT * FROM product_allergens")
allergens_data = db.run("SELECT * FROM allergens")
promotions_data = db.run("SELECT * FROM promotions")
reviews_data = db.run("SELECT * FROM reviews")

# DataFrame 생성
df_stores = pd.DataFrame(ast.literal_eval(stores_data), columns=['store_id', 'store_name', 'address', 'city', 'phone', 'opened_date'])
df_products = pd.DataFrame(ast.literal_eval(products_data), columns=['product_id', 'product_name', 'category_id', 'price', 'description', 'image_url'])
df_inventory = pd.DataFrame(ast.literal_eval(inventory_data), columns=['inventory_id', 'store_id', 'product_id', 'quantity_in_stock', 'last_updated'])
df_categories = pd.DataFrame(ast.literal_eval(categories_data), columns=['category_id', 'category_name', 'description'])
df_store_hours = pd.DataFrame(ast.literal_eval(store_hours_data), columns=['store_hours_id', 'store_id', 'day_of_week', 'opening_time', 'closing_time'])
df_product_allergens = pd.DataFrame(ast.literal_eval(product_allergens_data), columns=['product_allergen_id', 'product_id', 'allergen_id'])
df_allergens = pd.DataFrame(ast.literal_eval(allergens_data), columns=['allergen_id', 'allergen_name'])
df_promotions = pd.DataFrame(ast.literal_eval(promotions_data), columns=['promotion_id', 'product_id', 'store_id', 'discount_percent', 'valid_from', 'valid_until'])
df_reviews = pd.DataFrame(ast.literal_eval(reviews_data), columns=['review_id', 'product_id', 'rating', 'comment', 'review_date'])

print("✅ 모든 데이터 로드 완료!\n")

# ============================================================================
# 1. 매장 기본 정보
# ============================================================================

print("📋 1. 매장 기본 정보")
table1_data = df_stores[['store_name', 'address', 'city', 'phone', 'opened_date']].copy()
table1_data.columns = ['매장명', '주소', '도시', '전화번호', '개점일자']

fig1 = go.Figure(data=[go.Table(
    header=dict(
        values=list(table1_data.columns),
        fill_color='paleturquoise',
        align='left',
        font=dict(size=12, color='black')
    ),
    cells=dict(
        values=[table1_data[col] for col in table1_data.columns],
        fill_color='lavender',
        align='left',
        font=dict(size=11)
    )
)])
fig1.update_layout(title='🏪 매장 기본 정보', height=400)
fig1.show()

# ============================================================================
# 2. 각 매장별 빵 재고수량, 상품 단가
# ============================================================================

print("\n📋 2. 각 매장별 빵 재고수량, 상품 단가")
table2_data = df_inventory.merge(df_stores[['store_id', 'store_name']], on='store_id')\
    .merge(df_products[['product_id', 'product_name', 'price']], on='product_id')\
    .sort_values(['store_name', 'product_name'])[['store_name', 'product_name', 'quantity_in_stock', 'price']]
table2_data.columns = ['매장명', '상품명', '재고수량', '단가(원)']

fig2 = go.Figure(data=[go.Table(
    header=dict(
        values=list(table2_data.columns),
        fill_color='lightgreen',
        align='left',
        font=dict(size=12, color='black')
    ),
    cells=dict(
        values=[table2_data[col] for col in table2_data.columns],
        fill_color='honeydew',
        align='left',
        font=dict(size=10)
    )
)])
fig2.update_layout(title='📦 각 매장별 빵 재고수량 및 상품 단가', height=800)
fig2.show()

# ============================================================================
# 3. 각 매장의 요일별 운영시간
# ============================================================================

print("\n📋 3. 각 매장의 요일별 운영시간")
table3_data = df_store_hours.merge(df_stores[['store_id', 'store_name']], on='store_id')\
    .sort_values(['store_name', 'store_hours_id'])[['store_name', 'day_of_week', 'opening_time', 'closing_time']]
table3_data.columns = ['매장명', '요일', '오픈시간', '마감시간']

fig3 = go.Figure(data=[go.Table(
    header=dict(
        values=list(table3_data.columns),
        fill_color='lightskyblue',
        align='left',
        font=dict(size=12, color='black')
    ),
    cells=dict(
        values=[table3_data[col] for col in table3_data.columns],
        fill_color='lightcyan',
        align='left',
        font=dict(size=10)
    )
)])
fig3.update_layout(title='🕐 각 매장의 요일별 운영시간', height=600)
fig3.show()

# ============================================================================
# 4. 상품별 알레르기 가능성 여부
# ============================================================================

print("\n📋 4. 상품별 알레르기 가능성 여부")

# 각 상품에 대한 알레르겐 목록 생성
allergen_list = []
for product_id in df_products['product_id'].unique():
    product_name = df_products[df_products['product_id'] == product_id]['product_name'].values[0]
    allergens = df_product_allergens[df_product_allergens['product_id'] == product_id]['allergen_id'].values
    if len(allergens) > 0:
        allergen_names = df_allergens[df_allergens['allergen_id'].isin(allergens)]['allergen_name'].tolist()
        allergen_str = ', '.join(allergen_names)
    else:
        allergen_str = '없음'
    allergen_list.append([product_id, product_name, allergen_str])

table4_data = pd.DataFrame(allergen_list, columns=['product_id', 'product_name', 'allergens'])
table4_data = table4_data[['product_name', 'allergens']]
table4_data.columns = ['상품명', '포함된 알레르겐']

fig4 = go.Figure(data=[go.Table(
    header=dict(
        values=list(table4_data.columns),
        fill_color='lightcoral',
        align='left',
        font=dict(size=12, color='black')
    ),
    cells=dict(
        values=[table4_data[col] for col in table4_data.columns],
        fill_color='mistyrose',
        align='left',
        font=dict(size=10)
    )
)])
fig4.update_layout(title='⚠️ 상품별 알레르겐 정보', height=800)
fig4.show()

# ============================================================================
# 5. 매장의 상품별 프로모션 정보 (할인율, 할인 기간)
# ============================================================================

print("\n📋 5. 매장의 상품별 프로모션 정보")
table5_data = df_promotions.merge(df_products[['product_id', 'product_name']], on='product_id')\
    .merge(df_stores[['store_id', 'store_name']], on='store_id', how='left')\
    .sort_values(['store_name', 'product_name'])

# store_name이 null인 경우 '전체 매장'으로 변경
table5_data['store_name'] = table5_data['store_name'].fillna('전체 매장')

table5_data = table5_data[['store_name', 'product_name', 'discount_percent', 'valid_from', 'valid_until']]
table5_data.columns = ['매장명', '상품명', '할인율(%)', '할인시작일', '할인종료일']

fig5 = go.Figure(data=[go.Table(
    header=dict(
        values=list(table5_data.columns),
        fill_color='gold',
        align='left',
        font=dict(size=12, color='black')
    ),
    cells=dict(
        values=[table5_data[col] for col in table5_data.columns],
        fill_color='lightyellow',
        align='left',
        font=dict(size=10)
    )
)])
fig5.update_layout(title='🎉 매장의 상품별 프로모션 정보', height=400)
fig5.show()

# ============================================================================
# 6. 상품에 대한 리뷰 정보 (평점, 리뷰 내용, 작성일)
# ============================================================================

print("\n📋 6. 상품에 대한 리뷰 정보")
table6_data = df_reviews.merge(df_products[['product_id', 'product_name']], on='product_id')\
    .sort_values(['product_name', 'review_date'], ascending=[True, False])[['product_name', 'rating', 'comment', 'review_date']]
table6_data.columns = ['상품명', '평점', '리뷰 내용', '작성일']

# 리뷰 내용이 너무 길 경우 줄바꿈
table6_data['리뷰 내용'] = table6_data['리뷰 내용'].str.slice(0, 50) + '...'

fig6 = go.Figure(data=[go.Table(
    header=dict(
        values=list(table6_data.columns),
        fill_color='plum',
        align='left',
        font=dict(size=12, color='black')
    ),
    cells=dict(
        values=[table6_data[col] for col in table6_data.columns],
        fill_color='thistle',
        align='left',
        font=dict(size=10),
        height=25
    )
)])
fig6.update_layout(title='⭐ 상품에 대한 리뷰 정보', height=1000)
fig6.show()

# ============================================================================
# 요약 정보
# ============================================================================

print("\n" + "="*80)
print("📊 데이터베이스 요약")
print("="*80)
print(f"✅ 총 매장 수: {len(df_stores)}개")
print(f"✅ 총 상품 수: {len(df_products)}개")
print(f"✅ 총 재고 건수: {len(df_inventory)}개")
print(f"✅ 운영시간 정보: {len(df_store_hours)}개")
print(f"✅ 알레르겐 매핑: {len(df_product_allergens)}개")
print(f"✅ 프로모션: {len(df_promotions)}개")
print(f"✅ 리뷰: {len(df_reviews)}개")
print("="*80)

✅ 모든 데이터 로드 완료!

📋 1. 매장 기본 정보



📋 2. 각 매장별 빵 재고수량, 상품 단가



📋 3. 각 매장의 요일별 운영시간



📋 4. 상품별 알레르기 가능성 여부



📋 5. 매장의 상품별 프로모션 정보



📋 6. 상품에 대한 리뷰 정보



📊 데이터베이스 요약
✅ 총 매장 수: 7개
✅ 총 상품 수: 30개
✅ 총 재고 건수: 210개
✅ 운영시간 정보: 49개
✅ 알레르겐 매핑: 96개
✅ 프로모션: 12개
✅ 리뷰: 30개
